Every thing I did to "improve" the baseline such as adding extra features or making a complex NN, has resulted in the WMAE score getting worse than the baseline resulting in a negative score. I am reverting back to my baseline and I am going to extensively test what improves the model, and what does not improve the model.

In [15]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import Ridge

import anndata as ad
import scanpy as sc
from scipy import sparse
import sys

In [16]:
SEED = 6
EMBED_DIM = 128        # gene embedding size from control cells
OUT_PCA_K = 64         # number of output PCA components
RIDGE_ALPHA = 1.0      # ridge strength
CTRL_CELL_SAMPLE = 15000  # sample control cells for speed/memory
np.random.seed(SEED)

In [17]:
from sklearn.model_selection import KFold
from myllia_metric import myllia_score

def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)

    out = {}
    for k in ["score", "wcos", "mean_term", "pred_wmae"]:
        out[k] = float(getattr(r, k))
    return out

def cv_eval(predict_fn, genes, dt_true, n_splits=8, seed=6):
    genes = np.asarray(genes)
    dt_true = dt_true.astype(np.float32, copy=False)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold = 0
    rows = []

    for tr_idx, va_idx in kf.split(genes):
        fold += 1
        genes_va = genes[va_idx].tolist()
        dt_va = dt_true[va_idx]

        dp_va = predict_fn(genes_va)
        s = score_delta(dt_va, dp_va)
        s["fold"] = fold
        s["n_val"] = int(len(va_idx))
        rows.append(s)

    scores = np.array([r["score"] for r in rows], dtype=float)
    return rows, float(scores.mean()), float(scores.std())

In [18]:
df_means = pd.read_csv('data/training_data_means.csv')
df_valmap = pd.read_csv("data/pert_ids_val.csv")

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]
baseline_mask = df_means["pert_symbol"] == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).values
missing_train = sorted({g for g in train_genes if g not in gene_columns})

X_train_means = df_train[gene_columns].to_numpy(dtype=np.float32) # Only gene columns
D_train = X_train_means - x_base[None, :] # Subtract baseline
n_train = D_train.shape[0] # Training perturbations

delta_baseline = D_train.mean(axis=0)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

adata = ad.read_h5ad("Data/training_cells.h5ad")

In [19]:
adata2 = adata.copy()
sc.pp.normalize_total(adata2, target_sum=1e4, inplace=True)

# subset to the 5127 output genes, keep order
varnames = adata2.var_names.astype(str)
gene_to_idx = {g: i for i, g in enumerate(varnames)}
ordered_idx = [gene_to_idx[g] for g in gene_columns if g in gene_to_idx]
adata2 = adata2[:, ordered_idx].copy()

X = adata2.X.tocsr(copy=True)
X.data = np.log2(X.data + 1.0).astype(np.float32)

obs = adata2.obs.copy()
obs["gene"] = obs["sgrna_symbol"].astype(str)
obs["ch"] = obs["channel"].astype(str)

# compute mean expression per (gene, channel)
# We'll do this efficiently with sparse sums
from collections import defaultdict

# map row indices for each group
groups = defaultdict(list)
for i, (g, ch) in enumerate(zip(obs["gene"].values, obs["ch"].values)):
    groups[(g, ch)].append(i)

# mean for each group
group_mean = {}
for key, idxs in groups.items():
    Xi = X[idxs]
    m = np.asarray(Xi.mean(axis=0)).ravel().astype(np.float32)
    group_mean[key] = m

channels = sorted(obs["ch"].unique().tolist())

# baseline per channel (non-targeting within that channel)
base_by_ch = {}
for ch in channels:
    key = ("non-targeting", ch)
    if key in group_mean:
        base_by_ch[ch] = group_mean[key]
    else:
        raise RuntimeError(f"Missing non-targeting in channel {ch}")

# build training rows: only the 80 training perturbation genes
train_genes_list = df_train["pert_symbol"].astype(str).tolist()

X_rows = []   # features
Y_rows = []   # delta vectors
G_rows = []   # gene label for GroupKFold

for g in train_genes_list:
    for ch in channels:
        key = (g, ch)
        if key not in group_mean:
            continue
        delta_gc = group_mean[key] - base_by_ch[ch]
        Y_rows.append(delta_gc)
        G_rows.append(g)
        # features filled later once you have gene embeddings

Y_gc = np.stack(Y_rows, axis=0).astype(np.float32)  # (n_samples, 5127)
G_gc = np.array(G_rows)
print("Channel-conditioned samples:", Y_gc.shape[0], "unique genes:", len(np.unique(G_gc)))


Channel-conditioned samples: 320 unique genes: 80


In [20]:
svd = TruncatedSVD(n_components=EMBED_DIM, random_state=SEED)
svd.fit(X)  # X: (n_cells, 5127)

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",128
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",6
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [21]:
gene_emb = svd.components_.T.astype(np.float32)
gene2emb = {g: gene_emb[i] for i, g in enumerate(gene_columns)}
emb_fallback = gene_emb.mean(axis=0)

In [22]:
ch_to_i = {ch:i for i,ch in enumerate(channels)}
Cdim = len(channels)

# we need the channel id for each sample in Y_gc in same order we created it
# rebuild it similarly
ch_list = []
for g in train_genes_list:
    for ch in channels:
        if (g, ch) in group_mean:
            ch_list.append(ch)
ch_arr = np.array(ch_list)

X_gc = np.hstack([
    np.vstack([gene2emb.get(str(g), emb_fallback) for g in G_gc]).astype(np.float32),
    np.eye(Cdim, dtype=np.float32)[[ch_to_i[c] for c in ch_arr]]
]).astype(np.float32)

In [23]:
from sklearn.model_selection import GroupKFold

def cv_channel_model(out_pca_k=64, ridge_alpha=1.0):
    gkf = GroupKFold(n_splits=8)
    rows = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_gc, Y_gc, groups=G_gc), 1):
        # fit on (gene,channel) samples
        Y_tr = Y_gc[tr_idx]
        X_tr = X_gc[tr_idx]
        Y_va = Y_gc[va_idx]
        X_va = X_gc[va_idx]

        out_pca = PCA(n_components=min(out_pca_k, Y_tr.shape[0] - 1), random_state=SEED)
        C_tr = out_pca.fit_transform(Y_tr)

        reg = Ridge(alpha=ridge_alpha, random_state=SEED)
        reg.fit(X_tr, C_tr)

        # predict (gene,channel) deltas on va samples
        C_hat = reg.predict(X_va).astype(np.float32)
        Y_hat = out_pca.inverse_transform(C_hat).astype(np.float32)

        # now collapse back to gene-level mean delta to score properly
        # Compare against D_train from means.csv for those genes
        genes_va = G_gc[va_idx]
        # average predicted deltas per gene across its channels in this fold
        pred_by_gene = {}
        true_by_gene = {}
        for g in np.unique(genes_va):
            pred_by_gene[g] = Y_hat[genes_va == g].mean(axis=0)

            # ground truth from training_data_means baseline, same as your D_train
            gi = np.where(train_genes == g)[0][0]
            true_by_gene[g] = D_train[gi]

        dp = np.stack([pred_by_gene[g] for g in pred_by_gene], axis=0).astype(np.float32)
        dt = np.stack([true_by_gene[g] for g in pred_by_gene], axis=0).astype(np.float32)

        s = score_delta(dt, dp)
        s["fold"] = fold
        s["n_val_genes"] = int(dt.shape[0])
        rows.append(s)

    df = pd.DataFrame(rows)
    print(df)
    print(f"[cv score] mean={df['score'].mean():.6f} std={df['score'].std():.6f}")
    return df

df_cv = cv_channel_model(out_pca_k=OUT_PCA_K, ridge_alpha=RIDGE_ALPHA)


      score      wcos  mean_term  pred_wmae  fold  n_val_genes
0  0.059653  0.446738   0.133530   0.091626     1           10
1  0.057581  0.342996   0.167877   0.072975     2           10
2  0.089286  0.425262   0.209956   0.067929     3           10
3  0.054373  0.413949   0.131351   0.079870     4           10
4  0.078918  0.441152   0.178890   0.087168     5           10
5  0.089193  0.443341   0.201184   0.076613     6           10
6  0.073142  0.462157   0.158263   0.081479     7           10
7  0.053630  0.406966   0.131779   0.096888     8           10
[cv score] mean=0.069472 std=0.015123


## Basline with proper CV
| fold | score   | wcos    | mean_term | pred_wmae | n_val |
|------|----------|----------|------------|------------|-------|
| 1    | 0.073252 | 0.440429 | 0.166319   | 0.082329   | 10    |
| 2    | 0.053580 | 0.395726 | 0.135397   | 0.080839   | 10    |
| 3    | 0.052177 | 0.360384 | 0.144781   | 0.069925   | 10    |
| 4    | 0.050815 | 0.432774 | 0.117417   | 0.099419   | 10    |
| 5    | 0.074993 | 0.458398 | 0.163598   | 0.085954   | 10    |
| 6    | 0.102337 | 0.505185 | 0.202573   | 0.080247   | 10    |
| 7    | 0.086168 | 0.405487 | 0.212506   | 0.068919   | 10    |
| 8    | 0.067786 | 0.401551 | 0.168809   | 0.087416   | 10    |

**CV score:** mean = 0.070138  
**Std:** 0.018137

